# Unit 11 solutions: neural reranking (learning to rank the candidate pool)

These reference solutions build the unit's learned second stage -- a tiny PyTorch MLP (`NeuralRerankerPath`) that re-scores a cheap retrieval **pool** -- and read every number honestly. Units 2-10 each ship a retrieval *path* that sees one slice of the signal; Unit 6 blended their calibrated scores with **fixed** weights, Unit 10 fused a sparse+dense pair. This unit keeps the two-stage shape -- cheap retrieval proposes a pool, an expensive model re-orders only that pool -- but replaces the fixed ordering key with a **learned** one built from a 17-dim feature vector per candidate.

Three honest headlines frame the work, and the solutions state all three plainly. First, the reranker only **re-orders the pool**, so retrieval sets a hard **recall ceiling** (~0.772 at pool 50) that no reranking can cross. Second, the **naive** training recipe **leaks and tanks**: label the reranker with a reader's train positives while the feature-paths were fit on that *same* train, and the paths have memorised those items -- the learned "high score => positive" rule is a train artifact that validation never shows, so it drops to ~0.288 hit@10, *below* plain score-order. The fix is a **time-ordered holdout inside train** (latest ~25% of a reader's positives = labels; earlier 75% = the profile every training-time feature is built from). That is the first of **two leak modes**; the second is subtler -- a *repeat-read* book's held-out read was being counted into the training popularity feature (`log_pop`), faking a +0.018 win (0.364 -> 0.346) until training popularity was counted from the label-item-excluded profile rows (Exercise 3). Third, the sobering result: with both leaks closed, the all-feature reranker (~0.346) only **ties** the fixed 6-way score-order (~0.348) and ~matches the two-tower (~0.340) -- a learned ranker does not automatically beat a careful fixed blend. The genuine lift is **content**: a content-only reranker (~0.380) beats the score-order, the all-feature model *dilutes* it, and `linear ~= MLP`. Coverage (~0.195) edges the two-tower but sits well below the Unit-6 blend, and the solutions read all of that as it is.

Everything is measured on the **validation** split (`val`); each reader's **seen** set is their train-positive history. The sealed `test` split stays sealed -- no cell here touches it. Training is seeded and CPU-deterministic (`seed=0`). To stay in budget the six retrieval paths are fit **once** and refit **once** on the 75% profile, and the leakage-safe feature matrix is assembled **once** (the ~15 s step); every model trained below fits on that cached matrix in well under a second. The solutions reuse `serving`, `profile_paths`, `path_factory`, `reranker`, `X` and `y` across the exercises -- nothing is refit.

In [1]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    NeuralRerankerPath,
    PathRegistry,
    PopularityRetrievalPath,
    RerankerModel,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    run_blended_scoreboard,
)
from bookrec.rerank import family_mask, feature_names, split_profile_labels
from bookrec.scoreboard import _seen_and_relevant

SEED = 0
K = 10                 # every scoreboard reports at the top-10 cutoff
POOL = 50              # pinned: the pool recall ceiling is ~0.772 @ pool 50 vs ~0.716 @ pool 30
PATH_NAMES = ("popularity", "lexical", "item-item", "mf", "semantic", "two-tower")

# --- the real interaction log + content (the same substrate as Units 1-10) -----------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")      # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")   # {item_id: "space joined tokens"}
glove = bookrec.load_glove_subset()                              # the gensim-free GloVe subset (Unit 7)

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])     # readers the personalized paths cannot serve

interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")       # plain row dicts the paths duck-type

# Each reader's train-positive history is their `seen` (exclusion) set. We grade on `val`.
seen_by = defaultdict(set)
for row in rows:
    if row["label"] == 1 and row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])


def fit_six(train_rows):
    """Fit the six retrieval paths whose calibrated scores feed the reranker (seed 0)."""
    return {
        "popularity": PopularityRetrievalPath().fit(train_rows, catalog=catalog_ids),
        "lexical": LexicalRetrievalPath(keywords).fit(train_rows, catalog=catalog_ids),
        "item-item": ItemItemRetrievalPath().fit(train_rows, catalog=catalog_ids),
        "mf": MatrixFactorizationPath(seed=SEED).fit(train_rows, catalog=catalog_ids),
        "semantic": SemanticEmbeddingRetrievalPath(keywords, glove).fit(train_rows, catalog=catalog_ids),
        "two-tower": TwoTowerRetrievalPath(seed=SEED).fit(train_rows, catalog=catalog_ids),
    }


# The SERVING paths: fit on the FULL train once. Every exercise reuses them (~60 s total; do once).
serving = fit_six(rows)

# The leakage-safe TRAINING profile: hold out each reader's latest ~25% of train positives as the
# reranker's LABELS; the earlier 75% is the PROFILE the feature-paths (and every content/popularity
# statistic) are built from. Refit the six paths on that 75% profile ONCE (shared below).
profile, labels = split_profile_labels(rows, 0.25)
profile_rows = [
    r
    for r in rows
    if not (
        r["split"] == "train"
        and int(r["label"]) == 1
        and int(r["item_id"]) in labels.get(int(r["reader_id"]), set())
    )
]
profile_paths = fit_six(profile_rows)


def path_factory(train_rows):
    """The profile refit is done once up front; the reranker reuses it (passed via train_paths)."""
    return profile_paths


# The shared reranker over the serving paths (pool=50). You assemble its training matrix in
# Exercise 3 and install a learned model in Exercise 4; later exercises reuse this one object.
reranker = NeuralRerankerPath(serving, path_factory, catalog, pool=POOL, seed=SEED)


def build_pool(paths, reader, seen, top=POOL):
    """A reader's candidate POOL: {item_id: {path_name: calibrated_score}} over the paths' top-`top`.

    `calibrate` maps each path's scores to [0, 1] so the six very different score scales become
    comparable; the pool is the UNION of the six calibrated top-`top` lists keyed by item id.
    """
    per_item = defaultdict(dict)
    context = {"seen": seen}
    for name in PATH_NAMES:
        path = paths[name]
        for cand in path.calibrate(path.retrieve(reader, context, top)):
            per_item[cand.item_id][name] = cand.score
    return per_item


def board(path):
    """hit@10 + catalog coverage on `val` via the single-path blended scoreboard (reproduces hit@k)."""
    result = run_blended_scoreboard(
        {path.name: path}, interactions_path,
        catalog_ids=catalog_ids, k=K, pool=K, cold_readers=cold_readers,
    )
    return result.hit_rate_at_k, result.catalog_coverage

## Exercise 1 - Build the candidate pool and read the 17-dim feature vector

The first stage, RETRIEVAL, is cheap and proposes a candidate **pool**: the union of the six paths' top-`POOL` calibrated lists (`build_pool`). The second stage, RANK, is expensive and precise -- it scores *only* that pool. Below I build one reader's pool, confirm its size lands between `POOL` and `6 * POOL` (the six lists overlap but never fully), and read the feature layout with `feature_names(PATH_NAMES)` -- 17 names that split into 6 per-path scores + 6 presence flags + 1 `n_paths` meta + 4 content signals. I let `NeuralRerankerPath` assemble the matrix itself; here I only read its column layout.

In [2]:
one_reader = sorted(seen_by)[0]          # the first reader with a train history

pool = build_pool(serving, one_reader, seen_by[one_reader])
pool_size = len(pool)                     # the union of the six paths' top-POOL lists
pool_in_range = bool(POOL <= pool_size <= 6 * POOL)

# Read the feature layout from the ordered names (the reranker assembles the matrix itself).
spec = feature_names(PATH_NAMES)          # the 17 ordered feature names
n_features = len(spec)                    # 17
score_names = {f"score:{p}" for p in PATH_NAMES}
presence_names = {f"in:{p}" for p in PATH_NAMES}
content_names = {"genre_frac", "genre_cos", "author_frac", "log_pop"}
n_score = sum(1 for s in spec if s in score_names)        # 6 per-path calibrated scores
n_presence = sum(1 for s in spec if s in presence_names)  # 6 presence flags
n_content = sum(1 for s in spec if s in content_names)    # 4 content signals
layout_adds_up = bool(n_score + n_presence + 1 + n_content == n_features == 17)

pool_explanation = (
    "Retrieve-then-rank splits the work in two: cheap RETRIEVAL proposes the pool (the union of the "
    "six paths' calibrated top-POOL lists), and the expensive RANK stage re-orders only that pool, "
    "scoring each candidate from its 17-dim feature vector. The reranker never sees a book retrieval "
    "left out, so the pool is the entire universe it can recommend from -- retrieval sets the ceiling."
)

print(f"reader {one_reader}: pool size = {pool_size}  (POOL={POOL} <= {pool_size} <= {6 * POOL}: {pool_in_range})")
print(f"feature layout: {n_features} wide = {n_score} score + {n_presence} presence + 1 meta + {n_content} content")
print("feature names:", spec)
print(pool_explanation)

assert pool_in_range, "the pool is the union of six top-POOL lists: between POOL and 6*POOL items"
assert n_features == 17, "the reranker reads a 17-dim feature vector per pooled candidate"
assert layout_adds_up, "17 = 6 per-path scores + 6 presence flags + 1 n_paths meta + 4 content"

reader 0: pool size = 163  (POOL=50 <= 163 <= 300: True)
feature layout: 17 wide = 6 score + 6 presence + 1 meta + 4 content
feature names: ['score:popularity', 'score:lexical', 'score:item-item', 'score:mf', 'score:semantic', 'score:two-tower', 'in:popularity', 'in:lexical', 'in:item-item', 'in:mf', 'in:semantic', 'in:two-tower', 'n_paths', 'genre_frac', 'genre_cos', 'author_frac', 'log_pop']
Retrieve-then-rank splits the work in two: cheap RETRIEVAL proposes the pool (the union of the six paths' calibrated top-POOL lists), and the expensive RANK stage re-orders only that pool, scoring each candidate from its 17-dim feature vector. The reranker never sees a book retrieval left out, so the pool is the entire universe it can recommend from -- retrieval sets the ceiling.


## Exercise 2 - The pool recall ceiling: retrieval caps the reranker

Because the reranker only re-orders the pool, it can recommend a relevant book **only if retrieval already put it in the pool**. The measure is **any-relevant-in-pool recall**: the fraction of eligible `val` readers whose pool holds at least one of their held-out relevant books. That number is a hard **ceiling** on hit@10. I compute it at pool 50 and pool 30 and confirm the bigger pool lifts it (~0.772 vs ~0.716) -- exactly why `POOL = 50` is pinned: a larger pool buys ~0.05 of ceiling for a tiny extra cost.

In [3]:
def pool_recall(paths, top):
    """Any-relevant-in-pool recall over the eligible `val` readers -- the reranker's hit@10 ceiling."""
    known = set(catalog_ids)
    seen_lookup, target_lookup = _seen_and_relevant(interactions_path, known, "val")
    cold = set(cold_readers)
    hits = []
    for reader in sorted(target_lookup):
        seen = seen_lookup[reader]
        relevant = target_lookup[reader] - seen
        if reader in cold or not relevant:
            continue
        per_item = build_pool(paths, reader, seen, top)
        hits.append(1.0 if any(i in relevant for i in per_item) else 0.0)
    return float(np.mean(hits)) if hits else 0.0


ceiling_50 = pool_recall(serving, 50)   # ~0.772
ceiling_30 = pool_recall(serving, 30)   # ~0.716
bigger_pool_lifts_ceiling = bool(ceiling_50 > ceiling_30)

ceiling_explanation = (
    "The pool recall ceiling is the fraction of eligible readers whose pool already contains a "
    "relevant book. The reranker only RE-ORDERS the pool, so a reader with no relevant book in "
    "their pool can never be a hit no matter how good the ranker is -- retrieval caps reranking, and "
    "hit@10 can never exceed this ceiling. A bigger pool raises the ceiling at more scoring cost."
)

print(f"pool recall ceiling: @50 = {ceiling_50:.3f}  |  @30 = {ceiling_30:.3f}  (bigger lifts: {bigger_pool_lifts_ceiling})")
print(ceiling_explanation)

assert bigger_pool_lifts_ceiling, "the larger pool-50 raises the recall ceiling above pool-30"
assert 0.70 <= ceiling_50 <= 0.85, "the measured pool-50 ceiling is ~0.772"
assert 0.65 <= ceiling_30 <= 0.80, "the measured pool-30 ceiling is ~0.716"

pool recall ceiling: @50 = 0.772  |  @30 = 0.716  (bigger lifts: True)
The pool recall ceiling is the fraction of eligible readers whose pool already contains a relevant book. The reranker only RE-ORDERS the pool, so a reader with no relevant book in their pool can never be a hit no matter how good the ranker is -- retrieval caps reranking, and hit@10 can never exceed this ceiling. A bigger pool raises the ceiling at more scoring cost.


## Exercise 3 - The leakage-safe time-ordered holdout

Where do the reranker's training labels come from? Not the pool directly -- retrieval *excludes* each reader's `seen` (their train positives), so the pool holds no positives to learn from. The fix is a **time-ordered holdout inside train** (`split_profile_labels`): order a reader's train positives by time, hold out the latest ~25% as the reranker's **labels**, and keep the earlier 75% as the **profile**. Every training-time input -- the feature-paths *and* the genre/author/popularity statistics -- is built from that 75% profile only.

One honest subtlety the solution reads carefully: the **real** leakage guarantee is not that the profile and label *sets* are disjoint, but that the training **pool excludes the profile** (`seen`), so a profile book is never even a training candidate. The sets themselves can overlap: a reader who read the same book at both an early and a late time has that book in both the 75% profile and the 25% labels. Measured here, ~1/3 of splittable readers have such a repeat-read overlap. The labels that actually become positive training rows are exactly `labels - profile`, which the pool-excludes-profile step enforces automatically.

That same overlap hid the **second leak**. A popularity count that keeps every train row whose book is in the reader's profile *set* also counts a repeat-read book's **held-out read** -- the event being predicted inflates its own `log_pop`. An earlier version of the reranker did exactly that and scored a phantom 0.364 (0.346 once fixed). The fix is event-level: every training statistic is counted from `profile_rows`, which drop **every occurrence** of a reader's held-out-label items. I check that no held-out-label event survives in `profile_rows`. I assemble the leakage-safe `(X, y)` once (the ~15 s step, which also fixes the serving statistics so `set_model` works afterward) and confirm the holdout is well-formed.

In [4]:
prof, lab = split_profile_labels(rows, 0.25)   # {reader: profile set}, {reader: label set}

# A reader with >= 2 train positives is splittable (profile + >=1 held-out label). Readers with
# < 2 positives keep all as profile and supply no label -- they never train the reranker.
label_readers = sorted(r for r in lab if lab[r] and prof.get(r))
some_reader = label_readers[0]
has_profile_and_label = bool(len(prof[some_reader]) >= 1 and len(lab[some_reader]) >= 1)

# The raw profile and label SETS are NOT always disjoint: a book re-read at both an early and a
# late time lands in both. Measure how common that is -- the honest number, not an assumption.
overlapping = [r for r in label_readers if (prof[r] & lab[r])]
overlap_fraction = len(overlapping) / len(label_readers)   # ~0.335
label_sets_always_disjoint = bool(len(overlapping) == 0)    # FALSE here -- repeat reads overlap

# The REAL leakage invariant: the labels that actually become positive rows are the held-out
# items the profile never contained. The training pool excludes the profile (seen), so these are
# exactly labels - profile; some_reader has at least one such genuinely-new held-out label.
new_labels = lab[some_reader] - prof[some_reader]
has_genuinely_held_out_label = bool(len(new_labels) >= 1)

# The SECOND leak's guard (event level): profile_rows (setup cell) drop EVERY occurrence of a
# held-out-label item, so no held-out read -- not even a repeat read of a profile book -- can enter
# a training popularity count.
no_label_event_in_profile_rows = not any(
    r["split"] == "train" and r["label"] == 1 and r["item_id"] in lab.get(r["reader_id"], set())
    for r in profile_rows
)

# Assemble the leakage-safe matrix ONCE (reused by every exercise below). It also records the
# reranker's full-train serving statistics, so set_model works in Exercise 4.
X, y = reranker.assemble_training_matrix(rows, train_paths=profile_paths)
width_is_17 = bool(X.shape[1] == 17)
has_positives = bool((y == 1.0).any())
has_negatives = bool((y == 0.0).any())

holdout_explanation = (
    "Retrieval excludes a reader's seen set, so the pool has no positives to learn from; the labels "
    "come from a time-ordered holdout INSIDE train. The earlier 75% is the profile every "
    "training-time feature and statistic is built from. Leakage-safety does NOT rely on the profile "
    "and label sets being disjoint -- a repeat-read book can sit in both. It relies on the training "
    "pool EXCLUDING the profile (seen), so a profile book is never a training candidate and every "
    "positive training row is a held-out label the feature-paths never saw as profile. Popularity "
    "must also be counted from label-item-excluded ROWS, not profile-set membership: otherwise a "
    "repeat read's held-out event inflates its own log_pop (a +0.018 phantom: 0.364 vs 0.346)."
)

print(f"splittable readers: {len(label_readers)}")
print(f"reader {some_reader}: profile={len(prof[some_reader])} labels={len(lab[some_reader])} "
      f"new (label - profile)={len(new_labels)}")
print(f"readers whose label set overlaps profile (repeat reads): {len(overlapping)} "
      f"({overlap_fraction:.3f}); label_sets_always_disjoint={label_sets_always_disjoint}")
print(f"no held-out-label event in profile_rows: {no_label_event_in_profile_rows}")
print(f"training matrix X: {X.shape} (17 wide: {width_is_17})  |  y has both classes: pos={has_positives} neg={has_negatives}")
print(f"positives={int((y == 1.0).sum())}  negatives={int((y == 0.0).sum())}")
print(holdout_explanation)

assert has_profile_and_label, "a splittable reader keeps a non-empty profile and at least one label"
assert has_genuinely_held_out_label, "the reader has >=1 held-out label never in its profile (the real leakage invariant)"
assert no_label_event_in_profile_rows, "profile_rows drop every held-out-label event (leak mode 2 closed)"
assert width_is_17, "the feature matrix is 17 columns wide"
assert has_positives and has_negatives, "y carries both pool positives and sampled pool negatives"

splittable readers: 540
reader 0: profile=12 labels=5 new (label - profile)=3
readers whose label set overlaps profile (repeat reads): 181 (0.335); label_sets_always_disjoint=False
no held-out-label event in profile_rows: True
training matrix X: (6875, 17) (17 wide: True)  |  y has both classes: pos=True neg=True
positives=625  negatives=6250
Retrieval excludes a reader's seen set, so the pool has no positives to learn from; the labels come from a time-ordered holdout INSIDE train. The earlier 75% is the profile every training-time feature and statistic is built from. Leakage-safety does NOT rely on the profile and label sets being disjoint -- a repeat-read book can sit in both. It relies on the training pool EXCLUDING the profile (seen), so a profile book is never a training candidate and every positive training row is a held-out label the feature-paths never saw as profile. Popularity must also be counted from label-item-excluded ROWS, not profile-set membership: otherwise a repeat r

## Exercise 4 - Train the reranker, register it, and read the val scoreboard

Now the learned ordering key. I fit a tiny MLP on the cached matrix over **all** feature families (a `BCEWithLogits` pointwise-logistic ranker with Adam -- under a second), install it with `reranker.set_model(...)`, and register the path (it owns `reranker-v1`). Then I read the `val` scoreboard on **both** hit@10 and catalog coverage against the best single path (two-tower), the U6 4-way blend, the U10 hybrid, and the fixed 6-way equal-weight **score-order** over the *same* pool. The gate is deliberately honest: the reranker must **not tank** (`>= two_tower - 0.01`) and must **tie** the score-order over its own pool within noise (`>= six_way - 0.01`). It does not beat it -- ~0.346 vs ~0.348 -- and its coverage (~0.195) edges the two-tower but is a real loss against the U6 blend.

In [5]:
# Train the clean-recipe reranker on the cached leakage-safe matrix and install it as the key.
all_mask = family_mask(("scores", "presence", "meta", "content"))
all_model = RerankerModel(seed=SEED).fit(X, y, feature_mask=all_mask)
reranker.set_model(all_model)             # install the learned ordering key on the shared reranker

registry = PathRegistry()
registry.register(reranker)
registered_name = reranker.artifact_name()   # "reranker-v1"

# The val scoreboard: hit@10 AND catalog coverage for every comparator.
reranker_hit, reranker_cov = board(reranker)
two_tower_hit, two_tower_cov = board(serving["two-tower"])

hybrid = HybridRetrievalPath(serving["lexical"], serving["two-tower"], weight=0.7, pool=50).fit(rows)
hybrid_hit, hybrid_cov = board(hybrid)

u6 = run_blended_scoreboard(
    {n: serving[n] for n in ("popularity", "lexical", "item-item", "mf")},
    interactions_path, catalog_ids=catalog_ids, k=K, pool=30,
    weights={"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0},
    cold_readers=cold_readers,
)
u6_hit, u6_cov = u6.hit_rate_at_k, u6.catalog_coverage

six_way = run_blended_scoreboard(
    {n: serving[n] for n in PATH_NAMES},
    interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL, cold_readers=cold_readers,
)
six_way_hit = six_way.hit_rate_at_k       # fixed equal-weight score-order over the same pool

not_tanked = bool(reranker_hit >= two_tower_hit - 0.01)       # gate (a)
ties_score_order = bool(reranker_hit >= six_way_hit - 0.01)    # gate (b): a TIE within noise
coverage_is_a_loss = bool(reranker_cov < u6_cov)               # honest: below the U6 blend

scoreboard_reading = (
    "With both leaks closed, the all-feature reranker only TIES the fixed 6-way score-order over its "
    "own pool and roughly matches the two-tower on hit@10 -- a learned ranker over all 17 features "
    "does not beat a careful fixed blend here. Its coverage edges the two-tower's but sits well below "
    "the U6 blend: it concentrates its picks on a narrower, on-taste slice of the catalog. Treating "
    "'we added a neural reranker' as an automatic win would be exactly the overclaim to avoid."
)

print(f"registered as: {registered_name}")
print(f"reranker   hit@10={reranker_hit:.3f}  cov={reranker_cov:.3f}")
print(f"two-tower  hit@10={two_tower_hit:.3f}  cov={two_tower_cov:.3f}")
print(f"U10 hybrid hit@10={hybrid_hit:.3f}  cov={hybrid_cov:.3f}")
print(f"U6 blend   hit@10={u6_hit:.3f}  cov={u6_cov:.3f}")
print(f"6-way score-order (pool {POOL}) hit@10={six_way_hit:.3f}")
print(f"gate: not_tanked={not_tanked}  ties_score_order={ties_score_order}  coverage_is_a_loss={coverage_is_a_loss}")
print(scoreboard_reading)

assert registered_name == "reranker-v1", "the path owns the registry name reranker-v1"
assert not_tanked, "gate (a): the clean reranker is not tanked (hit@10 >= two-tower - 0.01)"
assert ties_score_order, "gate (b): the all-feature reranker ties the fixed 6-way score-order (within 0.01)"
assert coverage_is_a_loss, "the reranker's catalog coverage sits below the U6 blend (honest cost)"

registered as: reranker-v1
reranker   hit@10=0.346  cov=0.195
two-tower  hit@10=0.340  cov=0.177
U10 hybrid hit@10=0.362  cov=0.192
U6 blend   hit@10=0.306  cov=0.333
6-way score-order (pool 50) hit@10=0.348
gate: not_tanked=True  ties_score_order=True  coverage_is_a_loss=True
With both leaks closed, the all-feature reranker only TIES the fixed 6-way score-order over its own pool and roughly matches the two-tower on hit@10 -- a learned ranker over all 17 features does not beat a careful fixed blend here. Its coverage edges the two-tower's but sits well below the U6 blend: it concentrates its picks on a narrower, on-taste slice of the catalog. Treating 'we added a neural reranker' as an automatic win would be exactly the overclaim to avoid.


## Exercise 5 - The leakage trap: the naive recipe tanks

Here is the headline mistake -- leak mode 1, path memorization. The **naive** recipe labels the reranker with a reader's train positives while the feature-paths were fit on that *same* full train -- so the paths have *memorised* the label items and score them inflatedly. The reranker dutifully learns "high score => positive", but that pattern is a training-set artifact: `val` candidates were never in any path's training set, so their scores are not inflated and the learned rule misfires. I reproduce it with `leaky=True` (this reranker does its own leaky fit; it does not touch the cached `X`), board it on `val`, and confirm it scores below both the clean reranker and the score-order.

In [6]:
leaky = NeuralRerankerPath(serving, path_factory, catalog, pool=POOL, seed=SEED, leaky=True)
leaky.fit(rows)                           # trains on the full-fit serving paths + full-train counts

leaky_hit, leaky_cov = board(leaky)       # ~0.288 (tanked)
# reranker_hit (clean recipe) and six_way_hit (fixed score-order) come from Exercise 4.
leaky_below_clean = bool(leaky_hit < reranker_hit)
leaky_below_score_order = bool(leaky_hit < six_way_hit)

leak_explanation = (
    "The feature-paths memorised the label items, so their in-train scores are inflated in a way "
    "val candidates never are -- a distribution shift between training and serving. The learned "
    "'high score => positive' rule reads that train artifact and misfires on val, so hit@10 tanks "
    "below plain score-order. The time-ordered holdout fixes it by building every training-time "
    "feature from the 75% profile only, so no path ever trained on a label item."
)

print(f"leaky (naive same-train) hit@10={leaky_hit:.3f}  cov={leaky_cov:.3f}")
print(f"  < clean reranker hit@10={reranker_hit:.3f} ({leaky_below_clean})")
print(f"  < 6-way score-order hit@10={six_way_hit:.3f} ({leaky_below_score_order})")
print(leak_explanation)

assert leaky_below_clean, "the leaky same-train recipe scores below the clean time-ordered-holdout recipe"
assert leaky_below_score_order, "the leaky recipe even falls below the plain fixed score-order"

leaky (naive same-train) hit@10=0.288  cov=0.444
  < clean reranker hit@10=0.346 (True)
  < 6-way score-order hit@10=0.348 (True)
The feature-paths memorised the label items, so their in-train scores are inflated in a way val candidates never are -- a distribution shift between training and serving. The learned 'high score => positive' rule reads that train artifact and misfires on val, so hit@10 tanks below plain score-order. The time-ordered holdout fixes it by building every training-time feature from the 75% profile only, so no path ever trained on a label item.


## Exercise 6 - Feature ablation: content carries the lift

Exercise 4 found a tie. Is there *any* genuine lift, and which features earn it? The counterintuitive answer: the **content** features, not the path scores. I retrain on two subsets (cheap -- the matrix is cached): **content-only** and **scores-only** (scores + presence + meta). Each installs on the shared `reranker` and boards. Content-only is the strongest variant (~0.380) and genuinely **beats the fixed 6-way score-order** (~0.348); scores-only falls back to about the score-order; and the all-feature reranker (~0.346) sits *below* content-only -- adding the correlated path scores **dilutes** the content signal. The reason: the generator's taste is author-following and genre-derived, a cross-reader signal the per-list min-max-calibrated path *scores* cannot carry. I restore the full-feature MLP afterward.

In [7]:
content_model = RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(("content",)))
scores_model = RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(("scores", "presence", "meta")))

# Install each model on the shared reranker, board val, then restore the full-feature MLP.
reranker.set_model(content_model)
content_hit, content_cov = board(reranker)   # ~0.380
reranker.set_model(scores_model)
scores_hit, scores_cov = board(reranker)     # ~ plain score-order
reranker.set_model(all_model)                # restore the full-feature MLP from Exercise 4

content_at_least_scores = bool(content_hit >= scores_hit)
content_beats_score_order = bool(content_hit >= six_way_hit)   # the genuine lift (six_way_hit from Ex 4)
all_feature_dilutes = bool(reranker_hit <= content_hit)        # recorded: the path scores dilute content

ablation_explanation = (
    "Content-only is the strongest reranker and clearly beats the fixed 6-way score-order; "
    "dropping the path-score features does not hurt -- adding them back DILUTES the content signal. "
    "The content features (genre and author affinity to the reader's history, log-popularity) "
    "express a cross-reader taste signal that the per-list min-max-calibrated path scores cannot "
    "carry. So a learned ranker beats a blend here only when it SEES new features -- not by "
    "re-combining the path scores a fixed blend already combines."
)

print(f"content-only hit@10={content_hit:.3f}  cov={content_cov:.3f}")
print(f"scores-only  hit@10={scores_hit:.3f}  cov={scores_cov:.3f}")
print(f"content-only >= scores-only: {content_at_least_scores}")
print(f"content-only >= 6-way score-order ({six_way_hit:.3f}): {content_beats_score_order}")
print(f"all-feature ({reranker_hit:.3f}) <= content-only (dilution, recorded): {all_feature_dilutes}")
print(ablation_explanation)

assert content_at_least_scores, "content-only reranking is at least as strong as scores-only (content carries the lift)"
assert content_beats_score_order, "content-only reranking beats the fixed 6-way score-order (the genuine lift)"

content-only hit@10=0.380  cov=0.116
scores-only  hit@10=0.346  cov=0.186
content-only >= scores-only: True
content-only >= 6-way score-order (0.348): True
all-feature (0.346) <= content-only (dilution, recorded): True
Content-only is the strongest reranker and clearly beats the fixed 6-way score-order; dropping the path-score features does not hurt -- adding them back DILUTES the content signal. The content features (genre and author affinity to the reader's history, log-popularity) express a cross-reader taste signal that the per-list min-max-calibrated path scores cannot carry. So a learned ranker beats a blend here only when it SEES new features -- not by re-combining the path scores a fixed blend already combines.


## Exercise 7 - Challenge: linear vs MLP (the non-linearity adds nothing)

If the lift is content *affinity* rather than a non-linear interaction of the path scores, a **linear** model should do as well as the MLP. I train a single-`Linear`-layer reranker over all feature families -- logistic regression, the Unit 6 blend's fixed weights now *learned* -- install it, and board it. It lands within ~0.1 of the MLP: `linear ~= MLP` on this data. This is recorded, never gated.

In [8]:
linear_model = RerankerModel(seed=SEED, linear=True).fit(X, y, feature_mask=all_mask)

reranker.set_model(linear_model)
linear_hit, linear_cov = board(reranker)
reranker.set_model(all_model)             # restore the full-feature MLP
# reranker_hit (the all-feature MLP) comes from Exercise 4.
linear_approx_mlp = bool(abs(linear_hit - reranker_hit) < 0.1)

linear_explanation = (
    "The single linear layer matches the MLP, so the non-linear hidden layer adds nothing here. The "
    "lift is a near-linear read of the content-affinity features, not a non-linear combination of "
    "correlated path scores -- which is exactly what the content ablation already implied."
)

print(f"linear hit@10={linear_hit:.3f}  |  MLP hit@10={reranker_hit:.3f}  |  |diff|={abs(linear_hit - reranker_hit):.3f}")
print(f"linear ~= MLP (within 0.1): {linear_approx_mlp}")
print(linear_explanation)

assert linear_approx_mlp, "linear ~= MLP within 0.1 (the non-linear combiner adds nothing here)"

linear hit@10=0.364  |  MLP hit@10=0.346  |  |diff|=0.018
linear ~= MLP (within 0.1): True
The single linear layer matches the MLP, so the non-linear hidden layer adds nothing here. The lift is a near-linear read of the content-affinity features, not a non-linear combination of correlated path scores -- which is exactly what the content ablation already implied.


## Exercise 8 - Challenge: the pool-size knob and the recall ceiling

The pool size is a knob with a clear trade: a bigger pool raises the recall **ceiling** the reranker can reach (Exercise 2), at the cost of more candidates to assemble features for and score per reader. I compute the ceiling across a few pool sizes, confirm it rises with the pool, and confirm the pinned pool-50 reranker's hit@10 sits **under** its pool-50 ceiling -- the cap no reranking can cross. The implication is where to spend effort: to raise hit@10 past the cap you must improve RETRIEVAL (a bigger or better pool), not the ranker.

In [9]:
pool_sizes = [20, 30, 50]
ceiling_by_pool = {p: pool_recall(serving, p) for p in pool_sizes}   # rises with p
ceiling_rises_with_pool = bool(ceiling_by_pool[20] <= ceiling_by_pool[50])

# reranker_hit (the pinned pool-50 reranker) comes from Exercise 4; it cannot beat its pool ceiling.
reranker_under_ceiling = bool(reranker_hit <= ceiling_by_pool[50])

pool_knob_explanation = (
    "A bigger pool lifts the recall ceiling but costs more feature assembly and scoring per reader. "
    "The reranker only re-orders the pool, so its hit@10 is capped by the pool's ceiling and can "
    "never cross it by reranking alone. To raise hit@10 past the cap you must improve RETRIEVAL -- a "
    "bigger or better candidate pool -- not spend more effort on the ranker."
)

for p in pool_sizes:
    print(f"pool {p:>2}: ceiling = {ceiling_by_pool[p]:.3f}")
print(f"ceiling rises with pool: {ceiling_rises_with_pool}")
print(f"pinned reranker hit@10={reranker_hit:.3f} <= pool-50 ceiling {ceiling_by_pool[50]:.3f}: {reranker_under_ceiling}")
print(pool_knob_explanation)

assert ceiling_rises_with_pool, "a bigger pool raises the recall ceiling (pool 20 <= pool 50)"
assert reranker_under_ceiling, "the reranker's hit@10 cannot exceed its pool-50 recall ceiling"

pool 20: ceiling = 0.626
pool 30: ceiling = 0.716
pool 50: ceiling = 0.772
ceiling rises with pool: True
pinned reranker hit@10=0.346 <= pool-50 ceiling 0.772: True
A bigger pool lifts the recall ceiling but costs more feature assembly and scoring per reader. The reranker only re-orders the pool, so its hit@10 is capped by the pool's ceiling and can never cross it by reranking alone. To raise hit@10 past the cap you must improve RETRIEVAL -- a bigger or better candidate pool -- not spend more effort on the ranker.
